# Step 4d — AP-10K Cross-Species Evaluation
DINOv2 + DINOv3 + SAM · full pipeline (multilayer + PCA + MNN) · diagonal PCK · per-species breakdown

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
RESULTS_DIR = f'{DRIVE_ROOT}/results/step4/ap10k'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')

In [ ]:
# Cell 1 — Install + clone + AP-10K download
!pip install -q timm pandas scikit-learn segment-anything
import os, subprocess, sys
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

AP10K_ROOT = f'{DRIVE_ROOT}/datasets/AP-10K'
AP10K_CHECK = os.path.join(AP10K_ROOT, 'images')
if not os.path.exists(AP10K_CHECK):
    print('AP-10K not found. Downloading ...')
    # Official source: https://github.com/AlexTheBad/AP-10K  — download via their instructions
    # Place images/ and annotations/ under AP10K_ROOT before running this cell
    print('Please download AP-10K from https://github.com/AlexTheBad/AP-10K')
    print(f'and place images/ and annotations/ under {AP10K_ROOT}')
else:
    print('AP-10K found.')
print('Ready.')

In [ ]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os, math
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from collections import defaultdict
from sklearn.decomposition import PCA

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

DINOV2_W   = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
DINOV3_W   = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
SAM_W      = f'{DRIVE_ROOT}/weights/sam_vit_b.pth'
FT_DINOV2  = f'{DRIVE_ROOT}/weights/finetuned/dinov2_best.pth'
FT_DINOV3  = f'{DRIVE_ROOT}/weights/finetuned/dinov3_best.pth'
THRESHOLDS = [0.05, 0.1, 0.2]
N_LAST_LAYERS = 3
PCA_COMPONENTS = 64
MNN_K = 5
MNN_T = 1.0
MNN_DIST = 1

In [ ]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

# AP-10K dataset
class AP10KDataset(Dataset):
    def __init__(self, root, split='test', split_num=1, min_visible_kps=4):
        self.root = root
        self.pairs = []
        ann_path = os.path.join(root, 'annotations', f'ap10k-{split}-split{split_num}.json')
        with open(ann_path, 'r') as f:
            data = json.load(f)
        id_to_img = {img['id']: img for img in data['images']}
        id_to_cat = {cat['id']: cat['name'] for cat in data['categories']}
        id_to_ann = {}
        for ann in data['annotations']:
            raw = ann['keypoints']
            n_vis = sum(1 for k in range(2, len(raw), 3) if raw[k] == 2)
            if n_vis >= min_visible_kps:
                id_to_ann[ann['image_id']] = ann
        cat_to_imgs = defaultdict(list)
        for img_id, ann in id_to_ann.items():
            if img_id in id_to_img:
                cat_to_imgs[ann['category_id']].append(id_to_img[img_id])
        for cat_id in cat_to_imgs:
            cat_to_imgs[cat_id].sort(key=lambda img: img['id'])
        for cat_id, imgs in cat_to_imgs.items():
            for i in range(0, len(imgs) - 1, 2):
                si = imgs[i]; ti = imgs[i + 1]
                sa = id_to_ann[si['id']]; ta = id_to_ann[ti['id']]
                def parse_kps(raw):
                    kps, ids = [], []
                    for j in range(0, len(raw), 3):
                        if raw[j+2] == 2:
                            kps.append([float(raw[j]), float(raw[j+1])])
                            ids.append(j // 3)
                    return kps, ids
                skps, sids = parse_kps(sa['keypoints'])
                tkps, tids = parse_kps(ta['keypoints'])
                shared = sorted(set(sids) & set(tids))
                if len(shared) < min_visible_kps:
                    continue
                sm = {k: p for p, k in enumerate(sids)}
                tm = {k: p for p, k in enumerate(tids)}
                self.pairs.append({
                    'src_path': os.path.join(root, 'images', si['file_name']),
                    'tgt_path': os.path.join(root, 'images', ti['file_name']),
                    'src_kps': [skps[sm[k]] for k in shared],
                    'tgt_kps': [tkps[tm[k]] for k in shared],
                    'kps_ids': shared,
                    'category': id_to_cat[cat_id],
                })
        self.normalize = Normalize(['src_img', 'trg_img'])
        print(f'AP-10K {split} split{split_num}: {len(self.pairs)} pairs')
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        src_img = read_img(pair['src_path'])
        tgt_img = read_img(pair['tgt_path'])
        sample = {
            'src_img':    src_img,
            'trg_img':    tgt_img,
            'src_kps':    torch.tensor(pair['src_kps']).float(),
            'trg_kps':    torch.tensor(pair['tgt_kps']).float(),
            'src_imsize': src_img.size(),
            'trg_imsize': tgt_img.size(),
            'category':   pair['category'],
            'kps_ids':    pair['kps_ids'],
            'src_imname': pair['src_path'],
            'trg_imname': pair['tgt_path'],
        }
        return self.normalize(sample)

def extract_dense_features(model, img_tensor):
    with torch.no_grad():
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def extract_dense_features_multilayer(model, img_tensor, n_last_layers=3):
    with torch.no_grad():
        layers = model.get_intermediate_layers(img_tensor, n=n_last_layers,
                                               return_class_token=False, norm=True)
        avg = torch.stack(layers, dim=0).mean(dim=0)
        B, N, D = avg.shape
        H = W = int(N ** 0.5)
        return avg.reshape(B, H, W, D)

def apply_pca_whitening(src_features, tgt_features, n_components=64):
    dev = tgt_features.device
    _, H, W, D = tgt_features.shape
    n_px = H * W
    tgt_np = tgt_features.squeeze(0).reshape(n_px, D).cpu().float().numpy()
    src_np = src_features.squeeze(0).reshape(n_px, D).cpu().float().numpy()
    k = min(n_components, n_px, D)
    pca = PCA(n_components=k, whiten=True)
    pca.fit(tgt_np)
    tgt_out = torch.from_numpy(pca.transform(tgt_np)).float().reshape(1, H, W, k).to(dev)
    src_out = torch.from_numpy(pca.transform(src_np)).float().reshape(1, H, W, k).to(dev)
    return src_out, tgt_out

def extract_dense_features_SAM(model, img_tensor, image_size=512):
    with torch.no_grad():
        img_r = F.interpolate(img_tensor, size=(image_size, image_size), mode='bilinear', align_corners=False)
        if image_size != 1024:
            orig_pe = model.image_encoder.pos_embed
            ng = image_size // 16
            pe_r = F.interpolate(orig_pe.permute(0,3,1,2), size=(ng,ng),
                                  mode='bicubic', align_corners=False).permute(0,2,3,1)
            model.image_encoder.pos_embed = nn.Parameter(pe_r, requires_grad=False)
        from torch.cuda.amp import autocast
        with autocast():
            emb = model.image_encoder(img_r)
        if image_size != 1024:
            model.image_encoder.pos_embed = orig_pe
        return emb.permute(0, 2, 3, 1)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def find_best_match_windowed_softargmax(similarities, width, height, K=5, temperature=1.0):
    idx = similarities.argmax().item()
    cx, cy = idx % width, idx // width
    x0 = max(cx - K // 2, 0); x1 = min(cx + K // 2 + 1, width)
    y0 = max(cy - K // 2, 0); y1 = min(cy + K // 2 + 1, height)
    window_sims = similarities.reshape(height, width)[y0:y1, x0:x1]
    weights = torch.softmax(window_sims.flatten() * temperature, dim=0)
    ys, xs = torch.meshgrid(
        torch.arange(y0, y1, dtype=torch.float32, device=similarities.device),
        torch.arange(x0, x1, dtype=torch.float32, device=similarities.device),
        indexing='ij'
    )
    return (weights * xs.flatten()).sum().item(), (weights * ys.flatten()).sum().item()

def compute_pck_ap10k(pred_points, gt_points, img_size, threshold):
    """PCK with diagonal normalization for AP-10K."""
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    W, H = img_size
    diag = math.sqrt(W ** 2 + H ** 2)
    nd   = dist / diag
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 4 — Load all models
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base
from src.models.segment_anything.segment_anything import sam_model_registry

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)
if os.path.exists(FT_DINOV2):
    c = torch.load(FT_DINOV2, map_location=device)
    dinov2.load_state_dict(c['model_state_dict'], strict=False)
    print('DINOv2 finetuned loaded.')
else:
    c = torch.load(DINOV2_W, map_location=device)
    dinov2.load_state_dict(c, strict=True)
    print('DINOv2 pretrained loaded.')
dinov2.eval()

try:
    from src.models.dinov3.dinov3.models.vision_transformer import vit_base as dinov3_vit_base
    dinov3 = dinov3_vit_base(
        img_size=(512, 512), patch_size=16,
        n_storage_tokens=4, layerscale_init=1.0, mask_k_bias=True
    ).to(device)
    if os.path.exists(FT_DINOV3):
        c3 = torch.load(FT_DINOV3, map_location=device)
        dinov3.load_state_dict(c3['model_state_dict'], strict=False)
        print('DINOv3 finetuned loaded.')
    else:
        c3 = torch.load(DINOV3_W, map_location=device)
        dinov3.load_state_dict(c3, strict=True)
        print('DINOv3 pretrained loaded.')
    dinov3.eval()
    DINOV3_OK = True
except Exception as e:
    print(f'DINOv3 unavailable ({e}). Skipping.')
    dinov3 = None
    DINOV3_OK = False

sam = sam_model_registry['vit_b'](checkpoint=SAM_W).to(device)
sam.eval()
print('SAM loaded.')

In [ ]:
# Cell 5 — Load AP-10K dataset
ap10k_test = AP10KDataset(AP10K_ROOT, split='test', split_num=1, min_visible_kps=4)
print(f'AP-10K test: {len(ap10k_test)} pairs')

In [ ]:
# Cell 6 — Full pipeline evaluation on AP-10K
# Pipeline: multilayer features -> PCA whitening -> MNN matching -> diagonal PCK

def evaluate_ap10k_full(model, dataset, device, thresholds, img_size, patch_size,
                          use_multilayer=True, n_last_layers=3,
                          use_pca=True, n_components=64,
                          use_mnn=True, K=5, temperature=1.0, max_patch_dist=1,
                          is_sam=False):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])

            if is_sam:
                sf = extract_dense_features_SAM(model, src_t, image_size=img_size)
                tf = extract_dense_features_SAM(model, tgt_t, image_size=img_size)
            elif use_multilayer:
                sf = extract_dense_features_multilayer(model, src_t, n_last_layers)
                tf = extract_dense_features_multilayer(model, tgt_t, n_last_layers)
            else:
                sf = extract_dense_features(model, src_t)
                tf = extract_dense_features(model, tgt_t)

            if use_pca and not is_sam:
                sf, tf = apply_pca_whitening(sf, tf, n_components)

            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()

            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, img_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)

                if use_mnn:
                    fwd_x, fwd_y = find_best_match_windowed_softargmax(sim, W, H, K, temperature)
                    tpx = min(max(int(round(fwd_x)), 0), W - 1)
                    tpy = min(max(int(round(fwd_y)), 0), H - 1)
                    tgt_patch = tf[0, tpy, tpx, :]
                    sf_flat = sf.reshape(H * W, D)
                    back_sim = F.normalize(tgt_patch.unsqueeze(0), dim=1) @ F.normalize(sf_flat, dim=1).T
                    back_idx = back_sim.squeeze(0).argmax().item()
                    back_x, back_y = back_idx % W, back_idx // W
                    dist = max(abs(back_x - px), abs(back_y - py))
                    if dist <= max_patch_dist:
                        mx, my = fwd_x, fwd_y
                    else:
                        sims_m = sim.clone()
                        sims_m[tpy * W + tpx] = -1e9
                        mx, my = find_best_match_windowed_softargmax(sims_m, W, H, K, temperature)
                else:
                    mx, my = find_best_match_windowed_softargmax(sim, W, H, K, temperature)

                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, img_size)
                preds.append([rx, ry])

            # AP-10K: diagonal PCK
            tgt_img_size = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_ap10k(preds, trg_kps.tolist(), tgt_img_size, thr)
                pcks[thr] = pck
            per_img.append({
                'category': sample['category'],
                'pck_scores': pcks,
                'src_imname': str(sample['src_imname'])
            })
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

print('=== DINOv2 full pipeline on AP-10K ===')
res_d2 = evaluate_ap10k_full(
    dinov2, ap10k_test, device, THRESHOLDS, img_size=518, patch_size=14,
    use_multilayer=True, n_last_layers=N_LAST_LAYERS,
    use_pca=True, n_components=PCA_COMPONENTS,
    use_mnn=True, K=MNN_K, temperature=MNN_T, max_patch_dist=MNN_DIST
)
stats_d2 = save_exp_results(res_d2, 'ap10k_dinov2_full', THRESHOLDS, RESULTS_DIR)

In [ ]:
# Cell 7 — DINOv3 + SAM on AP-10K
if DINOV3_OK:
    print('=== DINOv3 full pipeline on AP-10K ===')
    res_d3 = evaluate_ap10k_full(
        dinov3, ap10k_test, device, THRESHOLDS, img_size=512, patch_size=16,
        use_multilayer=True, n_last_layers=N_LAST_LAYERS,
        use_pca=True, n_components=PCA_COMPONENTS,
        use_mnn=True, K=MNN_K, temperature=MNN_T, max_patch_dist=MNN_DIST
    )
    stats_d3 = save_exp_results(res_d3, 'ap10k_dinov3_full', THRESHOLDS, RESULTS_DIR)
else:
    stats_d3 = None

print('\n=== SAM on AP-10K ===')
res_sam = evaluate_ap10k_full(
    sam, ap10k_test, device, THRESHOLDS, img_size=512, patch_size=16,
    use_multilayer=False, use_pca=False,
    use_mnn=True, K=MNN_K, temperature=MNN_T, max_patch_dist=MNN_DIST,
    is_sam=True
)
stats_sam = save_exp_results(res_sam, 'ap10k_sam_full', THRESHOLDS, RESULTS_DIR)

In [ ]:
# Cell 8 — Per-species breakdown (DINOv2 results)
from collections import defaultdict

cat_scores = defaultdict(list)
for m in res_d2:
    cat_scores[m['category']].append(m['pck_scores'][0.1])

species_rows = [
    {'Species': cat, 'N pairs': len(scores), 'PCK@0.10 (%)': f'{np.mean(scores):.2f}'}
    for cat, scores in sorted(cat_scores.items(), key=lambda x: -np.mean(x[1]))
]
df_species = pd.DataFrame(species_rows)
print('=== Per-species PCK@0.10 (DINOv2 full pipeline) ===')
print(df_species.to_string(index=False))
df_species.to_csv(f'{RESULTS_DIR}/per_species_dinov2.csv', index=False)
print(f'Saved per-species breakdown.')

In [ ]:
# Cell 9 — Ablation: pipeline components (DINOv2 on AP-10K)
configs = [
    {'name': 'LastLayer+Argmax',        'multilayer': False, 'pca': False, 'mnn': False},
    {'name': 'Multilayer+Argmax',       'multilayer': True,  'pca': False, 'mnn': False},
    {'name': 'Multilayer+PCA+Argmax',   'multilayer': True,  'pca': True,  'mnn': False},
    {'name': 'Multilayer+PCA+MNN',      'multilayer': True,  'pca': True,  'mnn': True},
]
ablation_rows = []
for cfg in configs:
    print(f"\n--- {cfg['name']} ---")
    res = evaluate_ap10k_full(
        dinov2, ap10k_test, device, [0.1], 518, 14,
        use_multilayer=cfg['multilayer'], n_last_layers=N_LAST_LAYERS,
        use_pca=cfg['pca'], n_components=PCA_COMPONENTS,
        use_mnn=cfg['mnn'], K=MNN_K, temperature=MNN_T, max_patch_dist=MNN_DIST
    )
    mean_pck = float(np.mean([m['pck_scores'][0.1] for m in res]))
    ablation_rows.append({'Pipeline': cfg['name'], 'PCK@0.10 (%)': f'{mean_pck:.2f}'})
    print(f'  PCK@0.10={mean_pck:.2f}%')

df_abl = pd.DataFrame(ablation_rows)
print('\n=== AP-10K Pipeline Ablation ===')
print(df_abl.to_string(index=False))
df_abl.to_csv(f'{RESULTS_DIR}/ap10k_ablation.csv', index=False)

In [ ]:
# Cell 10 — Summary table
rows = [
    {'Model': 'DINOv2 (full pipeline)', 'Stats': stats_d2},
    {'Model': 'DINOv3 (full pipeline)', 'Stats': stats_d3},
    {'Model': 'SAM (full pipeline)',    'Stats': stats_sam},
]
table = []
for r in rows:
    if r['Stats'] is None:
        table.append({'Model': r['Model'], 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})
    else:
        table.append({
            'Model': r['Model'],
            'PCK@0.05': f"{r['Stats'].get('pck@0.05',{}).get('mean',0):.2f}%",
            'PCK@0.10': f"{r['Stats'].get('pck@0.10',{}).get('mean',0):.2f}%",
            'PCK@0.20': f"{r['Stats'].get('pck@0.20',{}).get('mean',0):.2f}%",
        })
df = pd.DataFrame(table)
print('\n=== Step 4d: AP-10K Results (diagonal PCK) ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/step4d_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/step4d_summary.csv')